In [1]:
import torch
import torch.nn.functional as F
def ce_on_mask(logits: torch.Tensor, target_idx: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Cross-entropy over masked positions for categorical logits."""
    if target_idx.dtype != torch.long:
        target_idx = target_idx.long()
    mask_bool = mask.squeeze(1) > 0.5
    if mask_bool.sum() == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)
    logits_flat = logits.permute(0, 2, 3, 1).reshape(-1, logits.size(1))
    target_flat = target_idx.reshape(-1)
    mask_flat = mask_bool.reshape(-1)
    logits_sel = logits_flat[mask_flat]
    target_sel = target_flat[mask_flat]
    return F.cross_entropy(logits_sel, target_sel)

In [2]:

B,K,H,W = 2, 1024, 16, 16
logits = torch.zeros(B,K,H,W)          # uniform -> CE ≈ ln K
target = torch.randint(0, K, (B,H,W))
mask   = torch.zeros(B,1,H,W); mask[:, :, :H//2, :W//2] = 1.0   # some masked, some visible

# Test A: uniform baseline
loss_u = ce_on_mask(logits, target, mask).item()
print("Uniform CE ~ lnK?", loss_u)  # expect ~ 6.93 for K=1024

# Test B: perfect predictions should go ~0
logits_perfect = torch.full_like(logits, -50.0)
logits_perfect.scatter_(1, target.unsqueeze(1), 50.0)
loss_p = ce_on_mask(logits_perfect, target, mask).item()
print("Perfect CE ~ 0?", loss_p)  # expect ~ 0.0

# Test C: no masked pixels -> loss=0 (by design)
print("No-mask CE=0?", ce_on_mask(logits, target, torch.zeros_like(mask)).item())

Uniform CE ~ lnK? 6.931472301483154
Perfect CE ~ 0? 0.0
No-mask CE=0? 0.0
